# 0 · Install Dependencies

In [39]:
!pip install transformers datasets torch scipy scikit-learn tqdm -q

# 1 · Imports & Config

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
# Explicit imports — replaces `from transformers import *` in original model.py
from transformers import (
    BertTokenizer, RobertaTokenizer,
    BertModel, RobertaModel,
    BertConfig, RobertaConfig,
    BertPreTrainedModel,
    get_linear_schedule_with_warmup,
)
from scipy.stats import pearsonr
from sklearn.metrics import classification_report, jaccard_score
from tqdm.auto import tqdm
import pandas as pd

# PATHS
MODEL_DIR_PATH  = "./models/"
DATA_DIR_PATH   = "./data/"
OUTPUT_DIR_PATH = "./output/"
IEMOCAP_PATH    = "./iemocap_transcripts/iemocap_merged_all.csv"

# ── Model config ──────────────────────────────────────────────────────────────
# src/main_ori.py → SingleDatasetTrainer._set_model_args()
MODEL_ARCH     = "bert"
MODEL_NAME     = "bert-base-uncased"
MAX_LEN        = 128
BATCH_SIZE     = 16
CONTEXT_WINDOW = 3       # prior turns prepended as BERT Segment A
CLIP_GRAD      = 1.0
WARMUP_RATIO   = 0.1
UPDATE_FREQ    = 1
TASK           = "vad-from-categories"
LABEL_TYPE     = "multi"

# ── Training schedule (shared by both models) ─────────────────────────────────
FREEZE_EPOCHS   = 5      # epochs with encoder frozen
UNFREEZE_EPOCHS = 5      # epochs with all params unfrozen
LR_FREEZE       = 3e-3
LR_UNFREEZE     = 5e-6

In [2]:
# src/models/trainer.py → Trainer.set_device()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    print("CPU")

Apple Silicon MPS


# 2 · IEMOCAP Data Preparation

In [3]:
_KEEP_COLS = [
    "session", "dialog", "utterance_id", "speaker",
    "start_time", "text", "emotion", "valence", "arousal", "dominance",
]

raw = pd.read_csv(IEMOCAP_PATH)

# Drop 47 utterances with no EmoEvaluation entry; sort chronologically within each dialog
iemocap_df = (
    raw[_KEEP_COLS]
    .dropna(subset=["valence", "arousal", "dominance"])
    .sort_values(["dialog", "start_time"])
    .reset_index(drop=True)
)

print(f"Total utterances : {len(iemocap_df)}")
print(f"Sessions         : {sorted(iemocap_df['session'].unique())}")
print(f"Dialogs          : {iemocap_df['dialog'].nunique()}")
print(f"Speakers         : {sorted(iemocap_df['speaker'].unique())}")
print(f"\nVAD ranges (IEMOCAP annotator scale 1-5)")
print(iemocap_df[["valence", "arousal", "dominance"]].agg(["min", "max", "mean"]).round(3))
iemocap_df.head(5)

Total utterances : 10039
Sessions         : ['Session1', 'Session2', 'Session3', 'Session4', 'Session5']
Dialogs          : 151
Speakers         : ['F', 'M']

VAD ranges (IEMOCAP annotator scale 1-5)
      valence  arousal  dominance
min     1.000     1.00      0.500
max     5.500     5.00      5.000
mean    2.778     3.09      3.194


,session,dialog,utterance_id,speaker,start_time,text,emotion,valence,arousal,dominance
0,Session1,Ses01F_impro01,Ses01F_impro01_F000,F,6.2901,Excuse me.,neu,2.5,2.5,2.5
1,Session1,Ses01F_impro01,Ses01F_impro01_M000,M,7.5712,Do you have your forms?,fru,2.5,2.0,2.5
2,Session1,Ses01F_impro01,Ses01F_impro01_F001,F,10.0100,Yeah.,neu,2.5,2.5,2.5
3,Session1,Ses01F_impro01,Ses01F_impro01_M001,M,10.9266,Let me see them.,fru,2.5,2.0,2.5
4,Session1,Ses01F_impro01,Ses01F_impro01_F002,F,14.8872,Is there a problem?,neu,2.5,2.5,2.5


In [6]:
# ── Per-speaker overview ──────────────────────────────────────────────────────
iemocap_F = iemocap_df[iemocap_df["speaker"] == "F"].reset_index(drop=True)
iemocap_M = iemocap_df[iemocap_df["speaker"] == "M"].reset_index(drop=True)

print(f"Female (F): {len(iemocap_F)}  |  Male (M): {len(iemocap_M)}")
print("\nEmotion distribution — F")
print(iemocap_F["emotion"].value_counts().to_string())
print("\nEmotion distribution — M")
print(iemocap_M["emotion"].value_counts().to_string())

# ── Session-based train / dev / test split ────────────────────────────────────
# Sessions 1-3 = train, Session 4 = dev, Session 5 = test.
# All turns in a dialog belong to one split — no context leakage across splits.
TRAIN_SESSIONS = ["Session1", "Session2", "Session3"]
DEV_SESSIONS   = ["Session4"]
TEST_SESSIONS  = ["Session5"]

iemocap_train = iemocap_df[iemocap_df["session"].isin(TRAIN_SESSIONS)].reset_index(drop=True)
iemocap_dev   = iemocap_df[iemocap_df["session"].isin(DEV_SESSIONS)].reset_index(drop=True)
iemocap_test  = iemocap_df[iemocap_df["session"].isin(TEST_SESSIONS)].reset_index(drop=True)

print(f"\nTrain : {len(iemocap_train):>5} utterances  ({iemocap_train['dialog'].nunique()} dialogs)")
print(f"Dev   : {len(iemocap_dev):>5} utterances  ({iemocap_dev['dialog'].nunique()} dialogs)")
print(f"Test  : {len(iemocap_test):>5} utterances  ({iemocap_test['dialog'].nunique()} dialogs)")

Female (F): 4800  |  Male (M): 5239

Emotion distribution — F
emotion
xxx    1252
fru     820
neu     733
ang     589
sad     557
exc     443
hap     327
sur      56
fea      22
oth       1

Emotion distribution — M
emotion
xxx    1255
fru    1029
neu     975
exc     598
sad     527
ang     514
hap     268
sur      51
fea      18
oth       2
dis       2

Train :  5766 utterances  (90 dialogs)
Dev   :  2103 utterances  (30 dialogs)
Test  :  2170 utterances  (31 dialogs)


# 3 · Shared Model Infrastructure

In [7]:
# src/data/__init__.py → EmotionDataset.__getitem__()
# Tokenizer selection mirrors main_ori.py → SingleDatasetTrainer.load_tokenizer()
if MODEL_ARCH == "bert":
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
else:
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], max_length=MAX_LEN,
            padding="max_length", truncation=True, return_tensors="pt",
        )
        lbl = self.labels[idx]
        t = torch.tensor(lbl, dtype=torch.long if LABEL_TYPE == "single" else torch.float)
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0), t

In [8]:
# BERT [CLS] → Dropout(0.1) → Linear(768, 3)
# src/models/model.py → PretrainedLMModel (task='vad-regression')

class BertForVADRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert    = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(self.bert.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        # src/models/model.py → PretrainedLMModel.forward() lines 109-129
        _, pooled = self.bert(
            input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, return_dict=False,
        )
        return self.head(self.dropout(pooled))   # (B, 3)


def _load_stage1_encoder(model):
    """Warm-start encoder weights from Stage 1 EMD checkpoint (best_model.pt)."""
    # src/models/trainer.py → load_model_from_ckeckpoint() lines 290-316
    ckpt = torch.load(MODEL_DIR_PATH + "best_model.pt", map_location=DEVICE)
    remap = {}
    for k, v in ckpt.items():
        if k.startswith("pre_trained_lm."):
            remap["bert." + k[len("pre_trained_lm."):]] = v
        elif k.startswith("dropout."):
            remap[k] = v
        # head.* skipped — shape is incompatible with new head
    missing, unexpected = model.load_state_dict(remap, strict=False)
    print(f"  Encoder keys loaded : {len(remap)}")
    print(f"  Missing (new head)  : {missing}")
    print(f"  Unexpected          : {unexpected}")


def encode_with_context(turns: list, idx: int, tok, max_len: int = MAX_LEN):
    """
    BERT sentence-pair encoding for turn at `idx`.
    Segment A: up to CONTEXT_WINDOW prior turns within the same dialog.
    Segment B: current turn.
    Truncation drops from the longest segment first, preserving the current turn.
    """
    current   = turns[idx]
    ctx_turns = turns[max(0, idx - CONTEXT_WINDOW): idx]
    ctx_str   = " ".join(f"{t['speaker']}: {t['text']}" for t in ctx_turns)
    cur_str   = f"{current['speaker']}: {current['text']}"
    if ctx_str:
        return tok(ctx_str, cur_str, max_length=max_len,
                   padding="max_length", truncation=True, return_tensors="pt")
    return tok(cur_str, max_length=max_len,
               padding="max_length", truncation=True, return_tensors="pt")

In [9]:
# ── Shared training / evaluation helpers ──────────────────────────────────────
# Defined here so context and no-context sections have no cross-cell dependencies.

mse_loss = nn.MSELoss()


def _pearson_r(P, L):
    # src/models/trainer.py → Trainer._compute_vad_eval_metrics() lines 402-405
    return [pearsonr(P[:, i], L[:, i])[0] for i in range(3)]


def _print_eval(title, P, L):
    r_v, r_a, r_d = _pearson_r(P, L)
    print(f"  [{title}]  V={r_v:+.4f}  A={r_a:+.4f}  D={r_d:+.4f}  "
          f"mean={(r_v + r_a + r_d) / 3:.4f}")


# Context model — 4-item batches (ids, mask, token_type_ids, labels)

def _train_epoch_ctx(mdl, loader, opt):
    mdl.train()
    total = 0.0
    for ids, mask, tti, labels in tqdm(loader, desc="  train", leave=False):
        ids, mask, tti, labels = ids.to(DEVICE), mask.to(DEVICE), tti.to(DEVICE), labels.to(DEVICE)
        loss = mse_loss(mdl(ids, attention_mask=mask, token_type_ids=tti), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), CLIP_GRAD)
        opt.step(); opt.zero_grad()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def _collect_ctx(mdl, loader):
    """Return (P, L) numpy arrays; context is intact — full-dialog batches."""
    mdl.eval()
    preds, lbls = [], []
    for ids, mask, tti, labels in tqdm(loader, desc="  eval", leave=False):
        ids, mask, tti = ids.to(DEVICE), mask.to(DEVICE), tti.to(DEVICE)
        preds.append(mdl(ids, attention_mask=mask, token_type_ids=tti).cpu())
        lbls.append(labels)
    return torch.cat(preds).numpy(), torch.cat(lbls).numpy()


def _eval_ctx(mdl, loader):
    P, L = _collect_ctx(mdl, loader)
    return _pearson_r(P, L)


# No-context model — 3-item batches (ids, mask, labels)

def _train_epoch_noctx(mdl, loader, opt):
    mdl.train()
    total = 0.0
    for ids, mask, labels in tqdm(loader, desc="  train", leave=False):
        ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        loss = mse_loss(mdl(ids, attention_mask=mask), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), CLIP_GRAD)
        opt.step(); opt.zero_grad()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def _collect_noctx(mdl, loader):
    """Return (P, L) numpy arrays."""
    mdl.eval()
    preds, lbls = [], []
    for ids, mask, labels in tqdm(loader, desc="  eval", leave=False):
        ids, mask = ids.to(DEVICE), mask.to(DEVICE)
        preds.append(mdl(ids, attention_mask=mask).cpu())
        lbls.append(labels)
    return torch.cat(preds).numpy(), torch.cat(lbls).numpy()


def _eval_noctx(mdl, loader):
    P, L = _collect_noctx(mdl, loader)
    return _pearson_r(P, L)


# Shared two-phase training loop

def _two_phase_train(mdl, train_ldr, dev_ldr, train_fn, eval_fn, ckpt_name):
    """Two-phase training; saves best checkpoint by mean Pearson r on dev."""
    best_r = float("-inf")
    # src/main_ori.py → SingleDatasetTrainer.train() lines 363-376
    for phase, (n_ep, lr) in enumerate(
            [(FREEZE_EPOCHS, LR_FREEZE), (UNFREEZE_EPOCHS, LR_UNFREEZE)], start=1):
        if phase == 1:
            for p in mdl.bert.parameters(): p.requires_grad = False
            trainable = mdl.head.parameters()
            print("Phase 1: encoder frozen")
        else:
            for p in mdl.parameters(): p.requires_grad = True
            trainable = mdl.parameters()
            print("Phase 2: all params unfrozen")

        opt = torch.optim.AdamW(trainable, lr=lr)
        for ep in range(1, n_ep + 1):
            tr_loss       = train_fn(mdl, train_ldr, opt)
            r_v, r_a, r_d = eval_fn(mdl, dev_ldr)
            mean_r        = (r_v + r_a + r_d) / 3
            print(f"  Ph{phase} Ep{ep:>2}/{n_ep}  loss={tr_loss:.4f}  "
                  f"V={r_v:.4f}  A={r_a:.4f}  D={r_d:.4f}  mean={mean_r:.4f}")
            if mean_r > best_r:
                best_r = mean_r
                torch.save(mdl.state_dict(), MODEL_DIR_PATH + ckpt_name)
                print(f"  Saved {ckpt_name}")
    print(f"\nBest dev mean Pearson r = {best_r:.4f}")
    return best_r

# 4 · Context-Aware Model (`model_s2`)

In [10]:
# ── IEMOCAPContextDataset ─────────────────────────────────────────────────────
# Batches: (input_ids, attention_mask, token_type_ids, labels)
# Context is built from ALL turns in the same dialog (both speakers interleaved).
# Grouping by dialog ensures context never crosses session boundaries.

class IEMOCAPContextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len: int = MAX_LEN):
        self.max_len = max_len
        self.samples: list = []   # each entry: (turns, idx, label)

        for _, grp in df.groupby("dialog", sort=False):
            grp   = grp.sort_values("start_time").reset_index(drop=True)
            turns = [{"speaker": r.speaker, "text": r.text} for r in grp.itertuples()]
            lbls  = grp[["valence", "arousal", "dominance"]].values.tolist()
            for i in range(len(turns)):
                self.samples.append((turns, i, lbls[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        turns, i, label = self.samples[idx]
        enc = encode_with_context(turns, i, tokenizer, self.max_len)
        tti = enc.get("token_type_ids", torch.zeros(self.max_len, dtype=torch.long))
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            tti.squeeze(0),
            torch.tensor(label, dtype=torch.float),
        )

    def speakers(self):
        """Speaker label per sample, in the same order as DataLoader iteration."""
        return [turns[i]["speaker"] for turns, i, _ in self.samples]

In [11]:
ctx_train_ds = IEMOCAPContextDataset(iemocap_train)
ctx_dev_ds   = IEMOCAPContextDataset(iemocap_dev)
ctx_test_ds  = IEMOCAPContextDataset(iemocap_test)

ctx_train_ldr = DataLoader(ctx_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
ctx_dev_ldr   = DataLoader(ctx_dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
ctx_test_ldr  = DataLoader(ctx_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Context DataLoaders (CONTEXT_WINDOW={CONTEXT_WINDOW}, batch_size={BATCH_SIZE}):")
print(f"  train : {len(ctx_train_ldr)} batches  ({len(ctx_train_ds)} samples)")
print(f"  dev   : {len(ctx_dev_ldr)} batches  ({len(ctx_dev_ds)} samples)")
print(f"  test  : {len(ctx_test_ldr)} batches  ({len(ctx_test_ds)} samples)")

_ids, _mask, _tti, _lbl = next(iter(ctx_train_ldr))
print(f"\nBatch shapes — ids:{list(_ids.shape)}  tti unique={_tti.unique().tolist()} (0=ctx 1=cur)")

Context DataLoaders (CONTEXT_WINDOW=3, batch_size=16):
  train : 361 batches  (5766 samples)
  dev   : 132 batches  (2103 samples)
  test  : 136 batches  (2170 samples)

Batch shapes — ids:[16, 128]  tti unique=[0, 1] (0=ctx 1=cur)


In [19]:
# ── Token-budget diagnostic ───────────────────────────────────────────────────
# Measures how many tokens go to context (Seg A) vs current turn (Seg B) after
# truncation to MAX_LEN=128 across the training split.
# When tti has no 1s the sample was encoded single-sentence (no prior context).

_n_ctx, _n_cur, _truncated, _has_ctx = [], [], [], []

for _ids, _mask, _tti, _lbl in ctx_train_ds:
    _m = _mask.numpy().astype(bool)
    _t = _tti.numpy()
    _has_seg_b = bool(_t.any())

    if _has_seg_b:
        ctx_len = int((_m & (_t == 0)).sum()) - 2   # subtract [CLS] and [SEP]
        cur_len = int((_m & (_t == 1)).sum()) - 1   # subtract final [SEP]
    else:
        ctx_len = 0
        cur_len = int(_m.sum()) - 2                 # subtract [CLS] and [SEP]

    _n_ctx.append(max(ctx_len, 0))
    _n_cur.append(max(cur_len, 0))
    _truncated.append(int(_m.sum()) == MAX_LEN)
    _has_ctx.append(_has_seg_b)

_n_ctx = np.array(_n_ctx)
_n_cur = np.array(_n_cur)
_trunc = np.array(_truncated)
_hctx  = np.array(_has_ctx)

print(f"Samples analysed    : {len(_n_ctx):,}")
print(f"Has context (Seg A) : {_hctx.sum():,}  ({_hctx.mean()*100:.1f}%)")
print(f"Truncated (no pad)  : {_trunc.sum():,}  ({_trunc.mean()*100:.1f}%)")
print()
print("── Token counts (context samples only) ──────────────────────")
_cm = _hctx
print(f"  Context tokens    mean={_n_ctx[_cm].mean():.1f}  "
      f"median={np.median(_n_ctx[_cm]):.0f}  "
      f"p5={np.percentile(_n_ctx[_cm], 5):.0f}  "
      f"p95={np.percentile(_n_ctx[_cm], 95):.0f}")
print(f"  Current-turn tok  mean={_n_cur[_cm].mean():.1f}  "
      f"median={np.median(_n_cur[_cm]):.0f}  "
      f"p5={np.percentile(_n_cur[_cm], 5):.0f}  "
      f"p95={np.percentile(_n_cur[_cm], 95):.0f}")
_budget = MAX_LEN - 3  # [CLS] ctx [SEP] cur [SEP]
print(f"  Ctx+Cur mean      = {(_n_ctx[_cm] + _n_cur[_cm]).mean():.1f}  "
      f"(token budget = {_budget})")
print()
print("── Context-token distribution ────────────────────────────────")
for _lo, _hi in [(0, 0), (1, 9), (10, 19), (20, 39), (40, MAX_LEN)]:
    _n = ((_n_ctx >= _lo) & (_n_ctx <= _hi)).sum()
    print(f"  {_lo:>3}–{_hi:<3} ctx tokens : {_n:,}  ({_n / len(_n_ctx) * 100:.1f}%)")

Samples analysed    : 5,766
Has context (Seg A) : 5,676  (98.4%)
Truncated (no pad)  : 321  (5.6%)

── Token counts (context samples only) ──────────────────────
  Context tokens    mean=51.8  median=48  p5=21  p95=94
  Current-turn tok  mean=17.8  median=14  p5=4  p95=44
  Ctx+Cur mean      = 69.7  (token budget = 125)

── Context-token distribution ────────────────────────────────
    0–0   ctx tokens : 90  (1.6%)
    1–9   ctx tokens : 40  (0.7%)
   10–19  ctx tokens : 193  (3.3%)
   20–39  ctx tokens : 1,708  (29.6%)
   40–128 ctx tokens : 3,735  (64.8%)


In [12]:
print("Initialising model_s2 from best_model.pt ...")
model_s2 = BertForVADRegression().to(DEVICE)
_load_stage1_encoder(model_s2)

Initialising model_s2 from best_model.pt ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoder keys loaded : 199
  Missing (new head)  : ['head.weight', 'head.bias']
  Unexpected          : []


### Training — context model

In [ ]:
_two_phase_train(
    model_s2, ctx_train_ldr, ctx_dev_ldr,
    _train_epoch_ctx, _eval_ctx,
    ckpt_name="best_iemocap_context_model.pt",
)

In [13]:
# ── Test evaluation — context model ──────────────────────────────────────────
# Per-speaker results are computed by splitting the prediction array post-hoc.
# This ensures context is always built from full dialogs (both speakers),
# not from speaker-filtered subsets which would corrupt the context window.

model_s2.load_state_dict(torch.load(MODEL_DIR_PATH + "best_iemocap_context_model.pt", map_location=DEVICE))

P_ctx, L_ctx = _collect_ctx(model_s2, ctx_test_ldr)
spk_ctx = np.array(ctx_test_ds.speakers())
f_ctx   = spk_ctx == "F"

print("=" * 58)
print("  Context Model — Test Evaluation (session 5)")
print("=" * 58)
_print_eval("All     ", P_ctx,            L_ctx)
_print_eval("Speaker F", P_ctx[f_ctx],    L_ctx[f_ctx])
_print_eval("Speaker M", P_ctx[~f_ctx],   L_ctx[~f_ctx])
print("=" * 58)

  eval:   0%|          | 0/136 [00:00<?, ?it/s]

  Context Model — Test Evaluation (session 5)
  [All     ]  V=+0.7632  A=+0.5004  D=+0.5292  mean=0.5976
  [Speaker F]  V=+0.7520  A=+0.4446  D=+0.5319  mean=0.5762
  [Speaker M]  V=+0.7736  A=+0.5477  D=+0.5184  mean=0.6132


# 5 · No-Context Model (`model_noctx`)

In [14]:
# ── IEMOCAPDataset (no context) ───────────────────────────────────────────────
# Each turn encoded as a single sentence: "speaker: text".
# Batches: (input_ids, attention_mask, labels) — no token_type_ids.
# self.speakers stores per-sample speaker labels for post-hoc evaluation.

class IEMOCAPDataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len: int = MAX_LEN):
        self.max_len  = max_len
        self.texts    = [f"{r.speaker}: {r.text}" for r in df.itertuples()]
        self.labels   = df[["valence", "arousal", "dominance"]].values.tolist()
        self.speakers = df["speaker"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding="max_length", truncation=True, return_tensors="pt",
        )
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float),
        )

In [18]:
nc_train_ds = IEMOCAPDataset(iemocap_train)
nc_dev_ds   = IEMOCAPDataset(iemocap_dev)
nc_test_ds  = IEMOCAPDataset(iemocap_test)

nc_train_ldr = DataLoader(nc_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
nc_dev_ldr   = DataLoader(nc_dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
nc_test_ldr  = DataLoader(nc_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"No-context DataLoaders (batch_size={BATCH_SIZE}):")
print(f"  train : {len(nc_train_ldr)} batches  ({len(nc_train_ds)} samples)")
print(f"  dev   : {len(nc_dev_ldr)} batches  ({len(nc_dev_ds)} samples)")
print(f"  test  : {len(nc_test_ldr)} batches  ({len(nc_test_ds)} samples)")

_ids, _mask, _lbl = next(iter(nc_train_ldr))
print(f"\nBatch shapes — ids:{list(_ids.shape)}  labels:{list(_lbl.shape)}")

No-context DataLoaders (batch_size=16):
  train : 361 batches  (5766 samples)
  dev   : 132 batches  (2103 samples)
  test  : 136 batches  (2170 samples)

Batch shapes — ids:[16, 128]  labels:[16, 3]


In [16]:
# model_noctx is completely independent from model_s2 — fresh weights from best_model.pt
print("Initialising model_noctx from best_model.pt ...")
model_noctx = BertForVADRegression().to(DEVICE)
_load_stage1_encoder(model_noctx)

Initialising model_noctx from best_model.pt ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoder keys loaded : 199
  Missing (new head)  : ['head.weight', 'head.bias']
  Unexpected          : []


### Training — no-context model

In [ ]:
_two_phase_train(
    model_noctx, nc_train_ldr, nc_dev_ldr,
    _train_epoch_noctx, _eval_noctx,
    ckpt_name="best_iemocap_nocontext_model.pt",
)

In [17]:
# ── Test evaluation — no-context model ───────────────────────────────────────
model_noctx.load_state_dict(torch.load(MODEL_DIR_PATH + "best_iemocap_nocontext_model.pt", map_location=DEVICE))

P_nc, L_nc = _collect_noctx(model_noctx, nc_test_ldr)
f_nc = np.array(nc_test_ds.speakers) == "F"

print("=" * 58)
print("  No-Context Model — Test Evaluation (session 5)")
print("=" * 58)
_print_eval("All     ", P_nc,           L_nc)
_print_eval("Speaker F", P_nc[f_nc],    L_nc[f_nc])
_print_eval("Speaker M", P_nc[~f_nc],   L_nc[~f_nc])
print("=" * 58)

  eval:   0%|          | 0/136 [00:00<?, ?it/s]

  No-Context Model — Test Evaluation (session 5)
  [All     ]  V=+0.6721  A=+0.4732  D=+0.4982  mean=0.5479
  [Speaker F]  V=+0.6387  A=+0.4088  D=+0.4722  mean=0.5066
  [Speaker M]  V=+0.7028  A=+0.5260  D=+0.5062  mean=0.5783


# Ablation

In [23]:
torch.manual_seed(42)
np.random.seed(42)

In [24]:
# ── Ablation: context window size ─────────────────────────────────────────────
# Train one context model per CONTEXT_WINDOW value in [2, 3, 4, 5].
# w=3 reuses best_iemocap_context_model.pt (already trained above).
# Checkpoints saved to models/best_iemocap_ctx_w{N}.pt for w != 3.

_WINDOWS      = [2, 3, 4, 5]
_ablation_res = {}   # window → (r_v, r_a, r_d)

for _w in _WINDOWS:
    print(f"\n{'='*58}")
    print(f"  CONTEXT_WINDOW = {_w}")
    print(f"{'='*58}")

    if _w == 3:
        # Reuse baseline checkpoint and loaders — no retraining needed
        print("  (reusing best_iemocap_context_model.pt)")
        _abl_test_ds  = ctx_test_ds
        _abl_test_ldr = ctx_test_ldr
        _ckpt = "best_iemocap_context_model.pt"
    else:
        _ckpt = f"best_iemocap_ctx_w{_w}.pt"

        # Build datasets with window size _w
        class _AblDS(Dataset):
            _w_ = _w  # capture loop variable
            def __init__(self, df):
                self.samples = []
                for _, grp in df.groupby("dialog", sort=False):
                    grp   = grp.sort_values("start_time").reset_index(drop=True)
                    turns = [{"speaker": r.speaker, "text": r.text} for r in grp.itertuples()]
                    lbls  = grp[["valence", "arousal", "dominance"]].values.tolist()
                    for i in range(len(turns)):
                        self.samples.append((turns, i, lbls[i]))
            def __len__(self):
                return len(self.samples)
            def __getitem__(self, idx):
                turns, i, label = self.samples[idx]
                current   = turns[i]
                ctx_turns = turns[max(0, i - self._w_): i]
                ctx_str   = " ".join(f"{t['speaker']}: {t['text']}" for t in ctx_turns)
                cur_str   = f"{current['speaker']}: {current['text']}"
                if ctx_str:
                    enc = tokenizer(ctx_str, cur_str, max_length=MAX_LEN,
                                    padding="max_length", truncation=True, return_tensors="pt")
                else:
                    enc = tokenizer(cur_str, max_length=MAX_LEN,
                                    padding="max_length", truncation=True, return_tensors="pt")
                tti = enc.get("token_type_ids", torch.zeros(MAX_LEN, dtype=torch.long))
                return (enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0),
                        tti.squeeze(0), torch.tensor(label, dtype=torch.float))
            def speakers(self):
                return [turns[i]["speaker"] for turns, i, _ in self.samples]

        _abl_train_ds  = _AblDS(iemocap_train)
        _abl_dev_ds    = _AblDS(iemocap_dev)
        _abl_test_ds   = _AblDS(iemocap_test)
        _abl_train_ldr = DataLoader(_abl_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
        _abl_dev_ldr   = DataLoader(_abl_dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        _abl_test_ldr  = DataLoader(_abl_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        _abl_model = BertForVADRegression().to(DEVICE)
        _load_stage1_encoder(_abl_model)
        _two_phase_train(
            _abl_model, _abl_train_ldr, _abl_dev_ldr,
            _train_epoch_ctx, _eval_ctx,
            ckpt_name=_ckpt,
        )

    # Evaluate on test set
    _eval_model = BertForVADRegression().to(DEVICE)
    _eval_model.load_state_dict(torch.load(MODEL_DIR_PATH + _ckpt, map_location=DEVICE))
    P_abl, L_abl = _collect_ctx(_eval_model, _abl_test_ldr)
    spk_abl = np.array(_abl_test_ds.speakers())
    f_abl   = spk_abl == "F"

    _ablation_res[_w] = tuple(_pearson_r(P_abl, L_abl))
    _print_eval("All     ", P_abl,           L_abl)
    _print_eval("Speaker F", P_abl[f_abl],   L_abl[f_abl])
    _print_eval("Speaker M", P_abl[~f_abl],  L_abl[~f_abl])


  CONTEXT_WINDOW = 2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoder keys loaded : 199
  Missing (new head)  : ['head.weight', 'head.bias']
  Unexpected          : []
Phase 1: encoder frozen


  train:   0%|          | 0/361 [00:00<?, ?it/s]

...

  Ph2 Ep 5/5  loss=0.2915  V=0.7062  A=0.4740  D=0.4267  mean=0.5356
  Saved best_iemocap_ctx_w5.pt

Best dev mean Pearson r = 0.5356


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  eval:   0%|          | 0/136 [00:00<?, ?it/s]

  [All     ]  V=+0.7518  A=+0.4888  D=+0.5283  mean=0.5896
  [Speaker F]  V=+0.7487  A=+0.4469  D=+0.5303  mean=0.5753
  [Speaker M]  V=+0.7557  A=+0.5261  D=+0.5222  mean=0.6014


In [25]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*58}")
print("  Ablation Summary — Test Pearson r  (Session 5)")
print(f"{'='*58}")
print(f"  {'Window':>8}  {'V':>7}  {'A':>7}  {'D':>7}  {'Mean':>7}")
print(f"  {'-'*44}")
for _w in _WINDOWS:
    r_v, r_a, r_d = _ablation_res[_w]
    mean_r = (r_v + r_a + r_d) / 3
    print(f"  w={_w}       {r_v:+.4f}  {r_a:+.4f}  {r_d:+.4f}  {mean_r:.4f}{tag}")
print(f"{'='*58}")


  Ablation Summary — Test Pearson r  (Session 5)
    Window        V        A        D     Mean
  --------------------------------------------
  w=2       +0.7361  +0.4825  +0.5309  0.5832
  w=3       +0.7632  +0.5004  +0.5292  0.5976
  w=4       +0.7583  +0.4947  +0.5462  0.5997
  w=5       +0.7518  +0.4888  +0.5283  0.5896
